In [ ]:
# Weights kernel (no submission; internet ON): desklib/ai-text-detector-v1.01 encoder (DeBERTa-v3-large fine-tuned as
# an AI-text detector) with the custom wrapper's 'model.' prefix stripped, saved as a plain DebertaV2Model + tokenizer
# in desklib-detector/ so the offline submission notebook can load it with AutoModel (find it by directory name: its
# config is model_type deberta-v2 / hidden 1024, the same as the plain DeBERTa-v3-large member).
import glob
import os

import pandas as pd
import torch
from huggingface_hub import snapshot_download
from safetensors.torch import load_file
from transformers import AutoConfig, AutoTokenizer, DebertaV2Model

d = snapshot_download('desklib/ai-text-detector-v1.01')
sd = {}
for f in glob.glob(os.path.join(d, '*.safetensors')):
    sd.update(load_file(f))
cfg = AutoConfig.from_pretrained(d)
model = DebertaV2Model(cfg)
res = model.load_state_dict({k[len('model.'):]: v for k, v in sd.items() if k.startswith('model.')}, strict=False)
miss = [k for k in res.missing_keys if 'pooler' not in k]
assert not miss, miss[:5]
model.save_pretrained('desklib-detector')
AutoTokenizer.from_pretrained(d).save_pretrained('desklib-detector')
print('saved desklib-detector', cfg.model_type, cfg.hidden_size, cfg.num_hidden_layers, 'missing', len(miss), flush=True)
pd.DataFrame({'id': [], 'model': []}).to_parquet('submission.parquet')   # placeholder
